# SITCOM on LDCT (practical demo)

This notebook provides a practical **LDCT image-domain denoising demo** using the SITCOM codebase.

Important notes:
- The official SITCOM paper/repo reports medical results on **MRI**, not LDCT.
- This notebook is an adaptation workflow to test feasibility on LDCT slices.
- For meaningful results, use a valid pretrained diffusion checkpoint (`models/ffhq_10m.pt` or your CT-domain checkpoint).

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import yaml
from PIL import Image
import matplotlib.pyplot as plt

try:
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-image", "-q"])
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity

repo_root = Path.cwd()
if not (repo_root / "guided_diffusion").exists():
    if (repo_root / "SITCOM" / "guided_diffusion").exists():
        repo_root = repo_root / "SITCOM"
    else:
        raise RuntimeError("Cannot find SITCOM root. Run this notebook inside SITCOM/ or workspace root.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from guided_diffusion.unet import create_model
from ddim_sampler import DDIMScheduler

torch.manual_seed(123)
np.random.seed(123)
print(f"repo_root = {repo_root}")

In [ ]:
# ==== User config ====
NDCT_DIR = repo_root / "data" / "ldct_demo" / "ndct"   # clean/full-dose CT PNG slices
LDCT_DIR = repo_root / "data" / "ldct_demo" / "ldct"   # low-dose CT PNG slices (optional)

MAX_IMAGES = 4
IMAGE_SIZE = 256
OUTPUT_DIR = repo_root / "results" / "ldct_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG_PATH = repo_root / "configs" / "model_config.yaml"
CKPT_PATH = repo_root / "models" / "ffhq_10m.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# SITCOM sampling hyper-parameters (smaller values for demo speed)
N_STEP = 8
NUM_STEPS = 8
LEARNING_RATE = 0.02
NOISELEVEL = 0.05

USE_EXISTING_LDCT = LDCT_DIR.exists() and len(list(LDCT_DIR.glob("*.png"))) > 0

print(f"DEVICE={DEVICE}")
print(f"NDCT_DIR={NDCT_DIR}")
print(f"LDCT_DIR={LDCT_DIR} (use_existing={USE_EXISTING_LDCT})")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CKPT_PATH}\n"
        "Download ffhq_10m.pt from the SITCOM README and place it under SITCOM/models/."
    )

In [ ]:
def list_pngs(folder: Path):
    return sorted([p for p in folder.glob("*.png")])


def load_gray_png(path: Path, image_size: int = 256):
    img = Image.open(path).convert("L").resize((image_size, image_size), Image.BICUBIC)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr


def simulate_ldct_from_ndct(ndct_01: np.ndarray, dose_scale: float = 0.25, gaussian_sigma: float = 0.01):
    lam = np.clip(ndct_01, 0.0, 1.0) * 255.0 * dose_scale
    poisson = np.random.poisson(lam) / (255.0 * dose_scale + 1e-8)
    noisy = poisson + np.random.normal(0.0, gaussian_sigma, size=ndct_01.shape)
    return np.clip(noisy, 0.0, 1.0).astype(np.float32)


def to_tensor_3ch_01(img_01: np.ndarray):
    t = torch.from_numpy(img_01).float()[None, None, :, :]
    return t.repeat(1, 3, 1, 1)


def to_minus1_1(x_01: torch.Tensor):
    return x_01 * 2.0 - 1.0


def to_01(x_m11: torch.Tensor):
    return ((x_m11.clamp(-1.0, 1.0) + 1.0) / 2.0)


def save_gray_png(img_01: np.ndarray, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.clip(img_01, 0.0, 1.0) * 255.0).astype(np.uint8), mode="L").save(path)


ndct_files = list_pngs(NDCT_DIR)
if len(ndct_files) == 0:
    raise RuntimeError(f"No PNG files found in {NDCT_DIR}")

print(f"Found {len(ndct_files)} NDCT slices")

In [ ]:
with open(MODEL_CONFIG_PATH, "r") as f:
    model_config = yaml.safe_load(f)

model_config["model_path"] = str(CKPT_PATH)
model = create_model(**model_config).to(DEVICE)
model.eval()

scheduler = DDIMScheduler()
scheduler.set_timesteps(num_inference_steps=N_STEP, device=DEVICE)

print("Model and scheduler are ready.")

In [ ]:
def sitcom_image_denoise(
    model,
    scheduler,
    y_obs_m11: torch.Tensor,
    n_step: int = 8,
    num_steps: int = 8,
    learning_rate: float = 0.02,
    noiselevel: float = 0.05,
):
    """
    A practical SITCOM-style image-domain denoising loop.
    y_obs_m11: observed LDCT tensor in [-1, 1], shape [1,3,H,W]
    """
    device = y_obs_m11.device
    step_size = 1000 // n_step

    alpha_last = scheduler.alphas_cumprod[-1].to(device)
    x_t = torch.randn_like(y_obs_m11)
    x_t = x_t * torch.sqrt(alpha_last) + torch.randn_like(y_obs_m11) * torch.sqrt(1.0 - alpha_last)

    product = int(y_obs_m11.shape[-2] * y_obs_m11.shape[-1] * y_obs_m11.shape[-3])

    pred_x0 = None
    for t in scheduler.timesteps:
        t_int = int(t.item())
        prev_timestep = t_int - step_size

        alpha_prod_t = scheduler.alphas_cumprod[t_int].to(device)
        if prev_timestep >= 0:
            alpha_prod_t_prev = scheduler.alphas_cumprod[prev_timestep].to(device)
        else:
            alpha_prod_t_prev = scheduler.alphas_cumprod[0].to(device)

        x_in = torch.nn.Parameter(x_t.clone())
        optimizer = torch.optim.Adam([x_in], lr=learning_rate)
        tt = (torch.ones(1, device=device) * t_int)

        for _ in range(num_steps):
            optimizer.zero_grad(set_to_none=True)
            noise_pred = model(x_in, tt)[:, :3]
            pred_x0 = (x_in - torch.sqrt(1.0 - alpha_prod_t) * noise_pred) / torch.sqrt(alpha_prod_t)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            # Identity measurement operator for image-domain LDCT denoising.
            loss = torch.norm(pred_x0 - y_obs_m11) ** 2
            if loss.item() < (noiselevel + 1e-3) ** 2 * product:
                break

            loss.backward()
            optimizer.step()

        with torch.no_grad():
            x_t = pred_x0.detach() * torch.sqrt(alpha_prod_t_prev) + torch.sqrt(1.0 - alpha_prod_t_prev) * torch.randn_like(x_t)

    return pred_x0.detach()

In [ ]:
records = []

for idx, ndct_path in enumerate(ndct_files[:MAX_IMAGES]):
    ndct_01 = load_gray_png(ndct_path, IMAGE_SIZE)

    if USE_EXISTING_LDCT:
        ldct_path = LDCT_DIR / ndct_path.name
        if not ldct_path.exists():
            print(f"Skip {ndct_path.name}: missing paired LDCT file")
            continue
        ldct_01 = load_gray_png(ldct_path, IMAGE_SIZE)
    else:
        ldct_path = None
        ldct_01 = simulate_ldct_from_ndct(ndct_01)

    ndct_t = to_minus1_1(to_tensor_3ch_01(ndct_01)).to(DEVICE)
    ldct_t = to_minus1_1(to_tensor_3ch_01(ldct_01)).to(DEVICE)

    start = time.time()
    den_t = sitcom_image_denoise(
        model=model,
        scheduler=scheduler,
        y_obs_m11=ldct_t,
        n_step=N_STEP,
        num_steps=NUM_STEPS,
        learning_rate=LEARNING_RATE,
        noiselevel=NOISELEVEL,
    )
    elapsed = time.time() - start

    den_01 = to_01(den_t).detach().cpu().numpy()[0, 0]

    psnr_noisy = peak_signal_noise_ratio(ndct_01, ldct_01, data_range=1.0)
    ssim_noisy = structural_similarity(ndct_01, ldct_01, data_range=1.0)
    psnr_den = peak_signal_noise_ratio(ndct_01, den_01, data_range=1.0)
    ssim_den = structural_similarity(ndct_01, den_01, data_range=1.0)

    out_name = ndct_path.stem + "_sitcom_ldct.png"
    out_path = OUTPUT_DIR / out_name
    save_gray_png(den_01, out_path)

    records.append({
        "name": ndct_path.name,
        "psnr_noisy": psnr_noisy,
        "psnr_denoised": psnr_den,
        "ssim_noisy": ssim_noisy,
        "ssim_denoised": ssim_den,
        "time_sec": elapsed,
        "output": str(out_path),
    })

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(ndct_01, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("NDCT (reference)")
    axes[0].axis("off")

    axes[1].imshow(ldct_01, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"LDCT/noisy\nPSNR={psnr_noisy:.2f}, SSIM={ssim_noisy:.3f}")
    axes[1].axis("off")

    axes[2].imshow(den_01, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"SITCOM demo\nPSNR={psnr_den:.2f}, SSIM={ssim_den:.3f}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

if len(records) == 0:
    print("No samples were processed. Check data paths and paired file names.")
else:
    avg_psnr_gain = np.mean([r["psnr_denoised"] - r["psnr_noisy"] for r in records])
    avg_ssim_gain = np.mean([r["ssim_denoised"] - r["ssim_noisy"] for r in records])

    print("\n==== Summary ====")
    for r in records:
        print(
            f"{r['name']}: "
            f"PSNR {r['psnr_noisy']:.2f} -> {r['psnr_denoised']:.2f}, "
            f"SSIM {r['ssim_noisy']:.3f} -> {r['ssim_denoised']:.3f}, "
            f"time={r['time_sec']:.2f}s"
        )

    print(f"\nAverage PSNR gain: {avg_psnr_gain:.3f} dB")
    print(f"Average SSIM gain: {avg_ssim_gain:.4f}")
    print(f"Saved outputs to: {OUTPUT_DIR}")

## Notes for stronger LDCT performance

- This demo uses an **image-domain identity measurement** objective for practicality.
- For physics-faithful LDCT, replace identity with a differentiable CT forward operator (projection + Poisson model) and use CT-domain pretrained diffusion priors.
- The released SITCOM checkpoint is FFHQ-domain; CT-domain retraining or finetuning is recommended for clinical-quality LDCT reconstruction.